# Eventi estremi SDE-Net t+6 con anomalie STGAN

Questo notebook non riaddestra né SDE-Net né STGAN. Usa la decisione `is_anomaly` già esportata da STGAN e la collega alle predizioni direct sull'esatto target `(location, timestamp)` con `horizon_hours == 6`.

La classificazione puntuale normale/anomala resta quella di STGAN: non viene stimata una nuova threshold. L'aggregazione delle località serve soltanto a individuare e visualizzare episodi regionali ed è quindi un'analisi post-hoc, non una parte del paper STGAN.

In [ ]:
import importlib, json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.reporting import anomaly_driver, anomaly_extremes, posthoc_outputs
from physiq_pv.reporting import mtgflow_spatiotemporal as spatial_context
anomaly_driver = importlib.reload(anomaly_driver)
anomaly_extremes = importlib.reload(anomaly_extremes)
posthoc_outputs = importlib.reload(posthoc_outputs)

STGAN_SEED = 20
FORECAST_HORIZON = 6
FOCUS_MONTHS = (4, 6, 7)
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR',
    ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}',
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_PATH',
    '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance/piedmont_pvgis_2019.nc',
)).resolve()
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT',
    ROOT / 'outputs' / f'sde_stgan_direct_multihorizon_seed{STGAN_SEED}',
)).resolve()
PREDICTIONS = EVALUATION_DIR / 'predictions.csv'
POSTHOC_DIR = EVALUATION_DIR / 'posthoc_by_horizon' / f't_plus_{FORECAST_HORIZON}'
REFERENCE_PEAK = POSTHOC_DIR / 'reference_production_peaks.csv'
FIGURE_DIR = EVALUATION_DIR / 'figures' / 'events' / 't_plus_6' / 'stgan_extreme_events'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print('STGAN scores       :', STGAN_SCORES)
print('Predizioni STGAN   :', PREDICTIONS)
print('Output eventi t+6  :', FIGURE_DIR)

## 1. Controllo delle sorgenti

Prima eseguire `stgan_pointwise_posthoc_sdenet.ipynb`: crea una copia evaluation-only delle predizioni SDE con le sole label STGAN e i post-hoc separati per orizzonte.

In [ ]:
required = (STGAN_SCORES, PREDICTIONS, EVALUATION_DIR / 'evaluation_source.json', REFERENCE_PEAK, PVGIS_2019)
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'File mancanti; eseguire prima stgan_pointwise_posthoc_sdenet.ipynb:\n'
        + '\n'.join(map(str, missing))
    )
metadata = json.loads((EVALUATION_DIR / 'evaluation_source.json').read_text(encoding='utf-8'))
if metadata.get('detector') != 'stgan':
    raise ValueError(f"Evaluation detector non STGAN: {metadata.get('detector')!r}")
score_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
if not {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'} <= score_header:
    raise ValueError('anomaly_scores.csv STGAN non ha lo schema atteso.')
prediction_header = set(pd.read_csv(PREDICTIONS, nrows=0).columns)
if 'event_group' in prediction_header:
    raise ValueError('Le predizioni contengono ancora event_group regionale: non sono STGAN-only.')
if not {'location', 'timestamp', 'horizon_hours', 'anomaly_group'} <= prediction_header:
    raise ValueError('predictions.csv evaluation-only non ha lo schema atteso.')
saved_horizons = sorted(pd.read_csv(PREDICTIONS, usecols=['horizon_hours'])['horizon_hours'].unique())
if FORECAST_HORIZON not in saved_horizons:
    raise ValueError(f't+{FORECAST_HORIZON} assente; orizzonti disponibili: {saved_horizons}')
display(pd.DataFrame([metadata]))
print('OK: label STGAN puntuali e forecast direct t+6 disponibili.')

## 2. Intensità regionale delle anomalie STGAN

Per ogni timestamp si misura la frazione di località con `is_anomaly=True`. La classifica giornaliera usa esclusivamente questi flag già salvati; lo score continuo serve solo come informazione secondaria per ordinare eventuali parità.

In [ ]:
regional = anomaly_extremes.regional_flag_series(STGAN_SCORES)
daily = anomaly_extremes.rank_flagged_days(regional)
focus_daily = daily.loc[daily['day'].dt.month.isin(FOCUS_MONTHS)].copy()
ranked_by_month = (
    focus_daily.assign(month=focus_daily['day'].dt.month)
    .sort_values(['month', 'n_anomalies', 'anomaly_share'], ascending=[True, False, False])
    .groupby('month', as_index=False, group_keys=False).head(10)
)
display(ranked_by_month[['month', 'day', 'n_anomalies', 'anomaly_share', 'n_active_hours', 'score_max']])
daily.to_csv(EVALUATION_DIR / 'stgan_daily_anomaly_ranking.csv', index=False)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharey=True)
for axis, month in zip(axes, FOCUS_MONTHS):
    month_data = focus_daily.loc[focus_daily['day'].dt.month == month].sort_values('day')
    axis.plot(month_data['day'], month_data['anomaly_share'], color='tab:red', lw=1.5)
    axis.fill_between(month_data['day'], month_data['anomaly_share'], alpha=0.2, color='tab:red')
    axis.set(title=f'STGAN — mese {month:02d} — decisioni salvate', ylabel='Frazione anomala')
    axis.grid(alpha=0.25)
axes[-1].set_xlabel('Timestamp target del forecast')
fig.suptitle('Anomalie regionali STGAN collegate alla valutazione SDE-Net t+6')
fig.tight_layout()
timeline_path = FIGURE_DIR / 'stgan_daily_anomaly_share_april_june_july_t_plus_6.png'
fig.savefig(timeline_path, dpi=180, bbox_inches='tight')
plt.show()

## 3. Giorni evento

Aprile e giugno mantengono i due use case fisici già definiti. Per luglio viene usato l'episodio regionale STGAN più intenso; se STGAN non produce un episodio regionale, viene riportato esplicitamente e si usa soltanto il giorno con il maggior numero di flag anomali per consentire l'ispezione, senza chiamarlo evento rilevato.

In [ ]:
episodes = anomaly_extremes.detect_extreme_episodes(regional, min_share='auto')
events = anomaly_extremes.group_episodes_into_events(episodes)
july_events = events.loc[events['days'].apply(
    lambda days: any(pd.Timestamp(day).month == 7 for day in days)
)] if not events.empty else events
july_events = july_events.sort_values(
    ['n_extreme_windows', 'duration_hours'], ascending=False
) if not july_events.empty else july_events
if not july_events.empty:
    july_days = tuple(day for day in july_events.iloc[0]['days'] if pd.Timestamp(day).month == 7)
    july_source = 'episodio regionale STGAN'
else:
    july_rank = focus_daily.loc[focus_daily['day'].dt.month == 7].sort_values(
        ['n_anomalies', 'anomaly_share'], ascending=False
    )
    if july_rank.empty:
        raise ValueError('STGAN non contiene timestamp di luglio 2019.')
    july_days = (july_rank.iloc[0]['day'].strftime('%Y-%m-%d'),)
    july_source = 'fallback: giorno STGAN più anomalo, non episodio regionale'

EVENT_WINDOWS = {
    'april_dust_23_26': ('2019-04-23', '2019-04-24', '2019-04-25', '2019-04-26'),
    'june_extreme_28_29': ('2019-06-28', '2019-06-29'),
    'july_stgan_peak': july_days,
}
print('Soglia regionale post-hoc:', episodes.attrs.get('min_share'))
print('Selezione luglio:', july_source, july_days)
display(events.loc[events['days'].apply(
    lambda days: any(pd.Timestamp(day).month in FOCUS_MONTHS for day in days)
)] if not events.empty else events)
display(pd.DataFrame([{'evento': name, 'giorni': ', '.join(days)} for name, days in EVENT_WINDOWS.items()]))

## 4. Distribuzione geografica STGAN

Ogni mappa mostra, per località, la frazione delle finestre che STGAN ha già classificato anomale durante l'evento. Non vengono ricostruite threshold dagli score e le coordinate entrano soltanto nella visualizzazione.

In [ ]:
locations, _, _ = spatial_context.load_pvgis_spatial_context(PVGIS_2019)
scores = pd.read_csv(STGAN_SCORES, usecols=['location', 'timestamp', 'is_anomaly'])
scores['location'] = scores['location'].astype(str)
scores['timestamp'] = pd.to_datetime(scores['timestamp'], errors='raise', utc=True).dt.tz_convert(None)
flag = scores['is_anomaly'].astype(str).str.strip().str.lower().map(
    {'true': True, 'false': False, '1': True, '0': False}
)
if flag.isna().any():
    raise ValueError('is_anomaly STGAN contiene valori non booleani.')
scores['is_anomaly'] = flag.astype(bool)
scores['day'] = scores['timestamp'].dt.normalize()
if not set(scores['location']).issubset(set(locations['location'])):
    raise ValueError('Alcune località STGAN non hanno coordinate PVGIS.')

map_rows = []
for event_name, event_days in EVENT_WINDOWS.items():
    normalized_days = pd.DatetimeIndex(pd.to_datetime(event_days)).normalize()
    aggregate = (
        scores.loc[scores['day'].isin(normalized_days)]
        .groupby('location', observed=True)['is_anomaly']
        .agg(anomaly_fraction='mean', n_scored='size', n_anomalies='sum')
        .reset_index()
    )
    aggregate['event'] = event_name
    map_rows.append(aggregate)
event_grid = pd.MultiIndex.from_product(
    [tuple(EVENT_WINDOWS), locations['location'].astype(str)], names=['event', 'location']
).to_frame(index=False)
event_map = (
    event_grid.merge(pd.concat(map_rows, ignore_index=True), on=['event', 'location'], how='left', validate='one_to_one')
    .merge(locations, on='location', how='left', validate='many_to_one')
)
event_map['anomaly_fraction'] = event_map['anomaly_fraction'].fillna(0.0)
event_map.to_csv(EVALUATION_DIR / 'stgan_event_location_anomaly_fraction.csv', index=False)

fig, axes = plt.subplots(1, len(EVENT_WINDOWS), figsize=(7 * len(EVENT_WINDOWS), 6), squeeze=False)
for axis, event_name in zip(axes.flat, EVENT_WINDOWS):
    frame = event_map.loc[event_map['event'].eq(event_name)]
    image = axis.scatter(
        frame['longitude'], frame['latitude'], c=frame['anomaly_fraction'],
        cmap='Reds', vmin=0.0, vmax=1.0, marker='s', s=24, linewidths=0,
    )
    axis.set(title=f'{event_name} — STGAN', xlabel='Longitudine', ylabel='Latitudine')
    axis.set_aspect(1.0 / np.cos(np.deg2rad(frame['latitude'].mean())))
    axis.grid(alpha=0.15)
fig.colorbar(image, ax=axes.ravel().tolist(), label='Frazione finestre STGAN anomale', shrink=0.8)
fig.suptitle('Distribuzione geografica degli eventi — target SDE-Net t+6')
spatial_path = FIGURE_DIR / 'stgan_geographic_event_maps_t_plus_6.png'
fig.savefig(spatial_path, dpi=180, bbox_inches='tight')
plt.show()

## 5. Errore SDE-Net t+6: normali contro anomalie STGAN

Le categorie anomale sono puntuali: una riga entra nel gruppo STGAN solo se la stessa località allo stesso timestamp target ha `is_anomaly=True`. Le metriche e i bin di produzione sono calcolati esclusivamente sul canale direct t+6.

In [ ]:
scores = scores.loc[scores['is_anomaly']].copy()
label_parts = []
for event_name, event_days in EVENT_WINDOWS.items():
    normalized_days = pd.DatetimeIndex(pd.to_datetime(event_days)).normalize()
    selected = scores.loc[scores['timestamp'].dt.normalize().isin(normalized_days), ['location', 'timestamp']].copy()
    selected['category'] = f'stgan:{event_name}'
    label_parts.append(selected)
event_labels = pd.concat(label_parts, ignore_index=True)
selected_days = tuple(sorted({day for days in EVENT_WINDOWS.values() for day in days}))
if event_labels.empty:
    raise ValueError('STGAN non marca alcuna riga anomala nei giorni selezionati.')

comparison = anomaly_driver.build_anomaly_driver_comparison_figures(
    EVALUATION_DIR, event_labels,
    figure_subdir='events/t_plus_6/stgan_extreme_events/pointwise',
    metrics_name='stgan_extreme_event_metrics_t_plus_6.csv',
    restrict_days=selected_days,
    figures_per_category=True,
    horizon_hours=FORECAST_HORIZON,
    reference_peak_path=REFERENCE_PEAK,
)
display(comparison['metrics'])
print('Metriche:', comparison['metrics_path'])

## 6. Confronto per finestra evento

Questo secondo confronto considera tutte le località nei giorni evento e le confronta con le righe che STGAN considera normali nel resto del 2019. Serve a distinguere l'impatto dell'evento fisico dalla sola sottopopolazione marcata anomala dal detector.

In [ ]:
calendar_results = {}
for event_name, event_days in EVENT_WINDOWS.items():
    calendar_results[event_name] = posthoc_outputs.build_extreme_event_comparison_figures(
        EVALUATION_DIR,
        event_dates=tuple(event_days),
        comparison_name=f'stgan_{event_name}_t_plus_6',
        figure_subdir=f'events/t_plus_6/stgan_extreme_events/{event_name}',
        horizon_hours=FORECAST_HORIZON,
        reference_peak_path=REFERENCE_PEAK,
    )
    print(event_name, '->', calendar_results[event_name]['metrics_path'])
    display(calendar_results[event_name]['metrics'])

## Lettura dei risultati

- Il confronto **pointwise** risponde: quanto sbaglia t+6 proprio dove STGAN segnala anomalia?
- Il confronto **calendar event** risponde: quanto sbaglia t+6 durante l'intero episodio fisico, comprese le località non flaggate?
- Se luglio usa il fallback, il notebook non afferma che STGAN abbia rilevato un episodio regionale: mostra soltanto il giorno più anomalo per consentire l'ispezione.
- Tutte le figure appaiono inline e sono salvate in `figures/events/t_plus_6/stgan_extreme_events`.